# Day 2 - Session 6: Exploratory Data Analysis - Cancer Genomics Dataset
**Duration: ~1.5 hours**

## Learning Objectives
- Apply R skills to real biomedical data
- Perform exploratory data analysis (EDA)
- Generate publication-quality visualizations
- Discover patterns and relationships in genomic data
- Practice the complete data analysis workflow

## 1. The Dataset: TCGA Breast Cancer

We'll work with clinical and molecular data from **The Cancer Genome Atlas (TCGA)** Breast Cancer study.

**Dataset includes:**
- 🧬 Gene expression data (selected genes)
- 👤 Patient clinical information
- 🏥 Treatment outcomes
- 🔬 Molecular subtypes

**Research Questions:**
1. How do gene expression levels differ between cancer subtypes?
2. Are there correlations between specific genes?
3. Do molecular subtypes have different survival rates?
4. Which genes best distinguish between subtypes?

## 2. Load Required Packages

In [ ]:
# Load packages
library(ggplot2)
library(dplyr)
library(tidyr)
library(pheatmap)  # For heatmaps

# If packages are missing, install:
# install.packages(c("ggplot2", "dplyr", "tidyr", "pheatmap"))

# Set seed for reproducibility
set.seed(123)

## 3. Create Simulated TCGA-like Dataset

We'll create a realistic simulated dataset based on TCGA characteristics.

In [ ]:
# Create simulated breast cancer dataset
n_samples <- 150

# Clinical data
clinical_data <- data.frame(
  patient_id = paste0("TCGA-", sprintf("%03d", 1:n_samples)),
  age = round(rnorm(n_samples, mean = 58, sd = 13)),
  tumor_size_cm = round(abs(rnorm(n_samples, mean = 3.5, sd = 1.5)), 1),
  subtype = sample(c("Luminal A", "Luminal B", "HER2+", "Basal-like"), 
                   n_samples, replace = TRUE, 
                   prob = c(0.4, 0.2, 0.15, 0.25)),
  stage = sample(c("I", "II", "III", "IV"), 
                 n_samples, replace = TRUE, 
                 prob = c(0.2, 0.4, 0.3, 0.1)),
  survival_months = round(abs(rnorm(n_samples, mean = 60, sd = 30))),
  status = sample(c("Alive", "Deceased"), 
                  n_samples, replace = TRUE, 
                  prob = c(0.7, 0.3))
)

head(clinical_data)

In [ ]:
# Gene expression data (key breast cancer genes)
# Expression values in log2(TPM+1) scale

gene_data <- data.frame(
  patient_id = clinical_data$patient_id,
  ESR1 = rnorm(n_samples, mean = 8, sd = 2),     # Estrogen receptor
  PGR = rnorm(n_samples, mean = 7, sd = 2.5),    # Progesterone receptor
  ERBB2 = rnorm(n_samples, mean = 6, sd = 1.5),  # HER2
  TP53 = rnorm(n_samples, mean = 9, sd = 1),     # Tumor suppressor
  MKI67 = rnorm(n_samples, mean = 7, sd = 2),    # Proliferation marker
  BRCA1 = rnorm(n_samples, mean = 6.5, sd = 1),  # DNA repair
  BRCA2 = rnorm(n_samples, mean = 6, sd = 1),    # DNA repair
  VEGFA = rnorm(n_samples, mean = 7.5, sd = 1.2) # Angiogenesis
)

# Add subtype-specific expression patterns
for(i in 1:n_samples) {
  subtype <- clinical_data$subtype[i]
  
  if(subtype == "Luminal A") {
    gene_data$ESR1[i] <- gene_data$ESR1[i] + 2
    gene_data$PGR[i] <- gene_data$PGR[i] + 1.5
    gene_data$MKI67[i] <- gene_data$MKI67[i] - 1
  } else if(subtype == "Luminal B") {
    gene_data$ESR1[i] <- gene_data$ESR1[i] + 1
    gene_data$MKI67[i] <- gene_data$MKI67[i] + 1.5
  } else if(subtype == "HER2+") {
    gene_data$ERBB2[i] <- gene_data$ERBB2[i] + 3
    gene_data$MKI67[i] <- gene_data$MKI67[i] + 1
  } else if(subtype == "Basal-like") {
    gene_data$ESR1[i] <- gene_data$ESR1[i] - 2
    gene_data$PGR[i] <- gene_data$PGR[i] - 2
    gene_data$MKI67[i] <- gene_data$MKI67[i] + 2
    gene_data$TP53[i] <- gene_data$TP53[i] + 1
  }
}

head(gene_data)

In [ ]:
# Merge clinical and gene expression data
cancer_data <- clinical_data %>%
  left_join(gene_data, by = "patient_id")

# Save for future use
write.csv(cancer_data, "breast_cancer_data.csv", row.names = FALSE)

print("Dataset created successfully!")
dim(cancer_data)
str(cancer_data)

## 4. Initial Data Exploration

In [ ]:
# Basic statistics
summary(cancer_data)

In [ ]:
# Distribution of subtypes
table(cancer_data$subtype)

# Distribution by stage
table(cancer_data$stage)

# Survival status
table(cancer_data$status)

In [ ]:
# Cross-tabulation: subtype vs stage
table(cancer_data$subtype, cancer_data$stage)

In [ ]:
# Summary statistics by subtype
cancer_data %>%
  group_by(subtype) %>%
  summarize(
    n = n(),
    mean_age = round(mean(age), 1),
    mean_tumor_size = round(mean(tumor_size_cm), 2),
    mean_survival = round(mean(survival_months), 1),
    pct_deceased = round(100 * mean(status == "Deceased"), 1)
  )

## 5. Visualizing Clinical Data

In [ ]:
# Distribution of subtypes
ggplot(cancer_data, aes(x = subtype, fill = subtype)) +
  geom_bar() +
  geom_text(stat = "count", aes(label = ..count..), vjust = -0.5) +
  labs(title = "Distribution of Breast Cancer Subtypes",
       x = "Molecular Subtype",
       y = "Number of Patients") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Age distribution by subtype
ggplot(cancer_data, aes(x = subtype, y = age, fill = subtype)) +
  geom_boxplot(alpha = 0.7) +
  geom_jitter(width = 0.2, alpha = 0.3) +
  labs(title = "Age Distribution by Breast Cancer Subtype",
       x = "Molecular Subtype",
       y = "Age (years)") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Tumor size vs survival
ggplot(cancer_data, aes(x = tumor_size_cm, y = survival_months, color = subtype)) +
  geom_point(size = 2, alpha = 0.6) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 1) +
  facet_wrap(~ subtype, nrow = 2) +
  labs(title = "Tumor Size vs Survival Time",
       x = "Tumor Size (cm)",
       y = "Survival (months)",
       color = "Subtype") +
  theme_minimal()

In [ ]:
# Survival by stage and subtype
ggplot(cancer_data, aes(x = stage, y = survival_months, fill = subtype)) +
  geom_boxplot() +
  labs(title = "Survival Time by Cancer Stage and Subtype",
       x = "Cancer Stage",
       y = "Survival (months)",
       fill = "Subtype") +
  theme_minimal()

## 6. Gene Expression Analysis

In [ ]:
# Summary of gene expression
gene_cols <- c("ESR1", "PGR", "ERBB2", "TP53", "MKI67", "BRCA1", "BRCA2", "VEGFA")

cancer_data %>%
  select(all_of(gene_cols)) %>%
  summary()

In [ ]:
# ESR1 (Estrogen Receptor) expression by subtype
ggplot(cancer_data, aes(x = subtype, y = ESR1, fill = subtype)) +
  geom_violin(alpha = 0.7) +
  geom_boxplot(width = 0.2, fill = "white", outlier.shape = NA) +
  labs(title = "ESR1 (Estrogen Receptor) Expression by Subtype",
       x = "Molecular Subtype",
       y = "ESR1 Expression (log2)") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# ERBB2 (HER2) expression by subtype
ggplot(cancer_data, aes(x = subtype, y = ERBB2, fill = subtype)) +
  geom_violin(alpha = 0.7) +
  geom_boxplot(width = 0.2, fill = "white", outlier.shape = NA) +
  labs(title = "ERBB2 (HER2) Expression by Subtype",
       subtitle = "Note high expression in HER2+ subtype",
       x = "Molecular Subtype",
       y = "ERBB2 Expression (log2)") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Reshape data for multi-gene comparison
gene_expression_long <- cancer_data %>%
  select(patient_id, subtype, all_of(gene_cols)) %>%
  pivot_longer(cols = all_of(gene_cols), 
               names_to = "gene", 
               values_to = "expression")

head(gene_expression_long)

In [ ]:
# Multiple genes comparison
ggplot(gene_expression_long, aes(x = subtype, y = expression, fill = subtype)) +
  geom_boxplot(alpha = 0.7) +
  facet_wrap(~ gene, scales = "free_y", nrow = 2) +
  labs(title = "Gene Expression Profiles Across Breast Cancer Subtypes",
       x = "Molecular Subtype",
       y = "Expression Level (log2)",
       fill = "Subtype") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")

## 7. Gene Correlation Analysis

In [ ]:
# Calculate correlation matrix
gene_matrix <- cancer_data %>%
  select(all_of(gene_cols)) %>%
  as.matrix()

cor_matrix <- cor(gene_matrix)
round(cor_matrix, 2)

In [ ]:
# Visualize correlation matrix as heatmap
pheatmap(cor_matrix,
         display_numbers = TRUE,
         number_format = "%.2f",
         color = colorRampPalette(c("blue", "white", "red"))(50),
         main = "Gene Expression Correlation Matrix")

In [ ]:
# Scatter plot: ESR1 vs PGR (hormone receptors)
ggplot(cancer_data, aes(x = ESR1, y = PGR, color = subtype)) +
  geom_point(size = 2, alpha = 0.6) +
  geom_smooth(method = "lm", se = TRUE, color = "black", linetype = "dashed") +
  labs(title = "Correlation: ESR1 vs PGR Expression",
       subtitle = "Estrogen and Progesterone receptors often co-expressed",
       x = "ESR1 Expression (log2)",
       y = "PGR Expression (log2)",
       color = "Subtype") +
  theme_minimal()

In [ ]:
# Test correlation significance
cor.test(cancer_data$ESR1, cancer_data$PGR)

## 8. Heatmap: Gene Expression Patterns

In [ ]:
# Prepare data for heatmap
heatmap_data <- cancer_data %>%
  arrange(subtype) %>%
  select(all_of(gene_cols))

# Create annotation for subtypes
annotation_row <- data.frame(
  Subtype = cancer_data %>% arrange(subtype) %>% pull(subtype)
)
rownames(annotation_row) <- cancer_data %>% arrange(subtype) %>% pull(patient_id)

# Create heatmap
pheatmap(t(heatmap_data),
         scale = "row",
         annotation_col = annotation_row,
         show_colnames = FALSE,
         cluster_cols = TRUE,
         cluster_rows = TRUE,
         main = "Gene Expression Heatmap by Patient",
         fontsize_row = 10)

## 9. Statistical Testing

In [ ]:
# ANOVA: Does ESR1 expression differ by subtype?
esr1_anova <- aov(ESR1 ~ subtype, data = cancer_data)
summary(esr1_anova)

In [ ]:
# Post-hoc test: Which subtypes differ?
TukeyHSD(esr1_anova)

In [ ]:
# T-test: ESR1 expression in Luminal A vs Basal-like
luminal_a <- cancer_data %>% filter(subtype == "Luminal A") %>% pull(ESR1)
basal <- cancer_data %>% filter(subtype == "Basal-like") %>% pull(ESR1)

t.test(luminal_a, basal)

## 10. Advanced Visualization: Combining Multiple Variables

In [ ]:
# Complex plot: Gene expression, survival, and subtype
ggplot(cancer_data, aes(x = MKI67, y = survival_months)) +
  geom_point(aes(color = subtype, size = tumor_size_cm), alpha = 0.6) +
  geom_smooth(method = "lm", se = TRUE, color = "black", linetype = "dashed") +
  scale_size_continuous(name = "Tumor Size (cm)", range = c(1, 5)) +
  labs(title = "MKI67 (Proliferation) vs Survival Time",
       subtitle = "Bubble size indicates tumor size",
       x = "MKI67 Expression (log2)",
       y = "Survival (months)",
       color = "Subtype") +
  theme_minimal()

In [ ]:
# Summary plot: Mean expression by subtype
gene_summary <- gene_expression_long %>%
  group_by(subtype, gene) %>%
  summarize(
    mean_expr = mean(expression),
    se = sd(expression) / sqrt(n()),
    .groups = "drop"
  )

ggplot(gene_summary, aes(x = gene, y = mean_expr, fill = subtype)) +
  geom_bar(stat = "identity", position = position_dodge()) +
  geom_errorbar(aes(ymin = mean_expr - se, ymax = mean_expr + se),
                position = position_dodge(0.9), width = 0.2) +
  labs(title = "Mean Gene Expression by Subtype",
       subtitle = "Error bars show standard error",
       x = "Gene",
       y = "Mean Expression (log2)",
       fill = "Subtype") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 11. Practice Exercise: Your Turn!

**Task:** Perform your own exploratory analysis

Choose one of these questions to investigate:
1. Does TP53 expression correlate with tumor size?
2. Which genes show the biggest difference between HER2+ and other subtypes?
3. Is there a relationship between BRCA1 expression and survival?
4. Create a visualization showing age, stage, and survival relationship

**Your analysis should include:**
- Summary statistics
- At least one visualization
- Statistical test (if appropriate)
- Brief interpretation

In [ ]:
# Your code here




## 12. Key Findings Summary

From our exploratory analysis, we discovered:

**Clinical Patterns:**
- ✅ Four distinct molecular subtypes with different frequencies
- ✅ Survival times vary by subtype and stage
- ✅ Tumor size shows weak correlation with survival

**Gene Expression:**
- ✅ ESR1 and PGR highly expressed in Luminal subtypes
- ✅ ERBB2 (HER2) elevated specifically in HER2+ subtype
- ✅ MKI67 (proliferation) highest in aggressive subtypes
- ✅ Basal-like tumors show distinct expression pattern

**Correlations:**
- ✅ ESR1 and PGR positively correlated (hormone receptors)
- ✅ Some genes cluster together in heatmap analysis
- ✅ Subtype-specific expression signatures identified

**Statistical Significance:**
- ✅ Significant differences in ESR1 expression between subtypes (p < 0.001)
- ✅ Post-hoc tests confirm specific subtype differences

**Clinical Relevance:**
These patterns reflect real biological differences in breast cancer and guide:
- Treatment selection (e.g., hormone therapy for ER+)
- Prognosis prediction
- Patient stratification for clinical trials

## 13. Complete EDA Workflow Recap

**Step 1: Load and Inspect** 📊
- Read data
- Check structure and dimensions
- Identify variable types

**Step 2: Clean and Prepare** 🧹
- Handle missing values
- Check for outliers
- Transform variables if needed

**Step 3: Summarize** 📈
- Descriptive statistics
- Frequency tables
- Group summaries

**Step 4: Visualize** 🎨
- Distributions (histograms, boxplots)
- Relationships (scatter plots, correlations)
- Comparisons (grouped plots, facets)

**Step 5: Test** 🔬
- Statistical tests
- Check assumptions
- Interpret p-values

**Step 6: Communicate** 💬
- Clear visualizations
- Concise summaries
- Actionable insights

## 14. Resources for Real Biomedical Data

**Public Genomics Databases:**
- **TCGA** (The Cancer Genome Atlas): https://portal.gdc.cancer.gov/
- **GEO** (Gene Expression Omnibus): https://www.ncbi.nlm.nih.gov/geo/
- **cBioPortal**: https://www.cbioportal.org/
- **ENCODE**: https://www.encodeproject.org/

**R Bioconductor Packages:**
- `TCGAbiolinks` - Download TCGA data
- `GEOquery` - Access GEO datasets
- `DESeq2` - Differential expression
- `edgeR` - RNA-seq analysis

**Practice Datasets:**
- Built-in R datasets: `data(package = "datasets")`
- `survival` package: Cancer survival data
- `MASS` package: Various medical datasets

**Learning Resources:**
- Bioconductor workflows: https://bioconductor.org/
- R for Data Science book: https://r4ds.had.co.nz/
- Modern Statistics for Modern Biology: https://www.huber.embl.de/msmb/

## Summary

In this session, you applied all your R skills to real biomedical data:
- ✅ Loaded and explored a complex cancer genomics dataset
- ✅ Performed comprehensive exploratory data analysis
- ✅ Created publication-quality visualizations
- ✅ Discovered biological patterns and relationships
- ✅ Conducted statistical tests
- ✅ Integrated clinical and molecular data

**You now have the skills to:**
- Analyze your own research data
- Generate figures for publications
- Perform reproducible analyses
- Explore complex biological datasets

---

## 🎉 Congratulations!

You've completed the R training course!

**What you've learned:**
- **Day 1**: R fundamentals, data manipulation, visualization
- **Day 2**: Best practices, dplyr, real-world biomedical data analysis

**Next steps:**
1. Apply these skills to your own research data
2. Explore specialized packages for your field
3. Join the R community and keep learning
4. Practice regularly to build fluency

**Remember:** The best way to learn R is by doing. Start with small analyses and build complexity over time.

**Happy coding! 🚀**